# Explore PI-CAI csPCa Label Folders

Three subfolders under `csPCa_lesion_delineations/human_expert/`:

| Folder | Description |
|--------|-------------|
| `original/` | Original expert annotations (may be at T2W, ADC, or DWI resolution). 1295/1500 cases. |
| `resampled/` | Same as `original/` but resampled to axial T2W resolution. 1295/1500 cases. |
| `Pooch25/` | New (2025) expert annotations for 205 cases that were previously AI-only. Binary labels (0/1). |

**Goal**: Understand which cases are in each folder, overlaps, and how to form a complete 1500-case set.

In [9]:
from pathlib import Path
import nibabel as nib
import numpy as np
import pandas as pd

BASE = Path("/home/bshi/Documents/prostate_train/data/raw/zenodo/picai/picai_labels/csPCa_lesion_delineations/human_expert")

orig_files    = sorted(BASE.glob("original/*.nii.gz"))
resampled_files = sorted(BASE.glob("resampled/*.nii.gz"))
pooch_files   = sorted(BASE.glob("Pooch25/*.nii.gz"))

orig_cases    = {f.name for f in orig_files}
resampled_cases = {f.name for f in resampled_files}
pooch_cases   = {f.name for f in pooch_files}

print(f"original:  {len(orig_cases)} files")
print(f"resampled: {len(resampled_cases)} files")
print(f"Pooch25:   {len(pooch_cases)} files")

original:  1295 files
resampled: 1295 files
Pooch25:   205 files


## Case overlap between folders

In [10]:
orig_only    = orig_cases - pooch_cases
pooch_only   = pooch_cases - orig_cases
both         = orig_cases & pooch_cases

print(f"In original only (not in Pooch25): {len(orig_only)}")
print(f"In Pooch25 only (not in original): {len(pooch_only)}")
print(f"In both original and Pooch25:      {len(both)}")
print()
print(f"original == resampled (same filenames): {orig_cases == resampled_cases}")
print()
union = orig_cases | pooch_cases
print(f"Union of original + Pooch25: {len(union)} cases")

In original only (not in Pooch25): 1295
In Pooch25 only (not in original): 205
In both original and Pooch25:      0

original == resampled (same filenames): True

Union of original + Pooch25: 1500 cases


## Cases in Pooch25 that overlap with original

In [11]:
print("Cases in BOTH original and Pooch25:")
for c in sorted(both):
    print(" ", c)

Cases in BOTH original and Pooch25:


## Unique cases in Pooch25 (not in original/resampled)

In [12]:
print(f"Pooch25-only cases ({len(pooch_only)}):")
for c in sorted(pooch_only):
    print(" ", c)

Pooch25-only cases (205):
  10008_1000008.nii.gz
  10013_1000013.nii.gz
  10029_1000029.nii.gz
  10036_1000036.nii.gz
  10044_1000044.nii.gz
  10050_1000050.nii.gz
  10053_1000053.nii.gz
  10094_1000094.nii.gz
  10100_1000100.nii.gz
  10104_1000104.nii.gz
  10106_1000106.nii.gz
  10112_1000112.nii.gz
  10132_1000134.nii.gz
  10135_1000137.nii.gz
  10140_1000142.nii.gz
  10148_1000150.nii.gz
  10170_1000173.nii.gz
  10171_1000174.nii.gz
  10182_1000185.nii.gz
  10209_1000213.nii.gz
  10212_1000216.nii.gz
  10214_1000218.nii.gz
  10225_1000229.nii.gz
  10229_1000233.nii.gz
  10232_1000236.nii.gz
  10235_1000239.nii.gz
  10236_1000240.nii.gz
  10271_1000275.nii.gz
  10273_1000278.nii.gz
  10280_1000285.nii.gz
  10282_1000288.nii.gz
  10283_1000289.nii.gz
  10293_1000299.nii.gz
  10312_1000318.nii.gz
  10318_1000324.nii.gz
  10324_1000330.nii.gz
  10334_1000340.nii.gz
  10354_1000360.nii.gz
  10358_1000364.nii.gz
  10365_1000371.nii.gz
  10369_1000375.nii.gz
  10371_1000377.nii.gz
  10375_

## Image dimensions

Sample a few files from each folder to compare shapes/voxel sizes.

In [13]:
def sample_shapes(files, n=5):
    rows = []
    for f in files[:n]:
        img = nib.load(f)
        rows.append({
            "file": f.name,
            "folder": f.parent.name,
            "shape": img.shape,
            "voxel_mm": tuple(np.round(img.header.get_zooms(), 3)),
            "unique_labels": sorted(np.unique(img.get_fdata()).astype(int).tolist()),
        })
    return pd.DataFrame(rows)

df_orig    = sample_shapes(orig_files)
df_resampled = sample_shapes(resampled_files)
df_pooch   = sample_shapes(pooch_files)

pd.concat([df_orig, df_resampled, df_pooch], ignore_index=True)

,file,folder,shape,voxel_mm,unique_labels
0,10000_1000000.nii.gz,original,"(116, 114, 31)","(1.754, 1.754, 3.3)",[0]
1,10001_1000001.nii.gz,original,"(120, 128, 21)","(2.0, 2.0, 3.6)",[0]
2,10002_1000002.nii.gz,original,"(120, 128, 22)","(2.0, 2.0, 3.0)",[0]
3,10003_1000003.nii.gz,original,"(120, 128, 23)","(2.0, 2.0, 3.0)",[0]
4,10004_1000004.nii.gz,original,"(120, 128, 21)","(2.0, 2.0, 3.6)",[0]
5,10000_1000000.nii.gz,resampled,"(640, 640, 31)","(0.281, 0.281, 3.3)",[0]
6,10001_1000001.nii.gz,resampled,"(640, 640, 21)","(0.3, 0.3, 3.6)",[0]
7,10002_1000002.nii.gz,resampled,"(384, 384, 22)","(0.5, 0.5, 3.0)",[0]
8,10003_1000003.nii.gz,resampled,"(384, 384, 23)","(0.5, 0.5, 3.0)",[0]
9,10004_1000004.nii.gz,resampled,"(640, 640, 21)","(0.3, 0.3, 3.6)",[0]


## Label values across all folders

Check unique labels in non-empty files (positive cases).

In [17]:
def label_summary(files, label="folder", n=20):
    all_labels = set()
    n_positive = 0
    sample = files[:n]
    for f in sample:
        data = nib.load(f).get_fdata()
        unique = set(np.unique(data).astype(int).tolist())
        if unique != {0}:
            n_positive += 1
        all_labels |= unique
    return {"folder": label, "n_sampled": len(sample), "n_positive_in_sample": n_positive,
            "all_unique_labels": sorted(all_labels)}

rows = [
    label_summary(orig_files, "original"),
    label_summary(resampled_files, "resampled"),
    label_summary(pooch_files, "Pooch25"),
]
df = pd.DataFrame(rows)
# Note: all_unique_labels reflects only the sample. Per README, full label range for
# original/resampled is {0, 2, 3, 4, 5} (ISUP granular; label 1 unused).
# Pooch25 uses binary {0, 1}.
df

,folder,n_sampled,n_positive_in_sample,all_unique_labels
0,original,20,4,"[0, 2, 3]"
1,resampled,20,4,"[0, 2, 3]"
2,Pooch25,20,20,"[0, 1]"


## Recommendation: which folder(s) to use for a complete 1500-case set?

**`original/` and `resampled/` cover the same 1295 cases** — the only difference is resolution:
- `original/`: native acquisition resolution (in-plane ~1.75–2 mm, variable matrices ~116–128 px)
- `resampled/`: resampled to axial T2W resolution (in-plane ~0.3–0.5 mm, large matrices ~384–640 px)

**`Pooch25/` covers the remaining 205 disjoint cases** (previously AI-only, now re-annotated by human experts). Notably, **all 205 Pooch25 cases are csPCa-positive** — these were selected specifically because they were the 205 positive cases lacking human annotation.

**For a complete 1500-case set**, combine `resampled/` + `Pooch25/`:
- Both are at T2W resolution — spatially compatible
- Covers all 1500 cases with zero overlap

**Label encoding differs**:
- `resampled/`: granular ISUP — 0 (benign/indolent), 2 (ISUP 2), 3, 4, 5. Label 1 is unused.
- `Pooch25/`: binary — 0 (benign/indolent), 1 (csPCa ≥ ISUP 2)

To make a unified binary csPCa mask: binarize `resampled/` as `label > 1`; use `Pooch25/` labels directly.

---

## Identifying cases with lesions

In `resampled/`, only ~17% of cases (220/1295) have lesions — the rest are all-zero masks. All 205 Pooch25 cases are positive.

**Option A — image-based** (ground truth from files, slow):
```python
resampled_positive = [f for f in resampled_files if np.any(nib.load(f).get_fdata() > 1)]
pooch_positive     = [f for f in pooch_files     if np.any(nib.load(f).get_fdata() == 1)]
# total positive = len(resampled_positive) + len(pooch_positive)  → expect 425
```

**Option B — marksheet.csv** (fast, no image loading, recommended):
```python
marksheet = pd.read_csv(BASE.parent.parent / "clinical_information/marksheet.csv")
positive_cases = marksheet[marksheet["lesion_GS"].notna()]
# positive_cases["study_id"] gives the case IDs with confirmed csPCa lesions
```

In [15]:
df_summary = pd.DataFrame([
    {"folder": "original",  "n_cases": len(orig_cases),      "in_plane_res": "~2 mm (native)",    "label_type": "ISUP granular (0,2-5)", "same_cases_as": "resampled"},
    {"folder": "resampled", "n_cases": len(resampled_cases), "in_plane_res": "~0.3-0.5 mm (T2W)", "label_type": "ISUP granular (0,2-5)", "same_cases_as": "original"},
    {"folder": "Pooch25",   "n_cases": len(pooch_cases),     "in_plane_res": "~0.3-0.5 mm (T2W)", "label_type": "binary (0,1), all positive", "same_cases_as": "none"},
])
print("Complete 1500-case set: resampled/ (1295 cases) + Pooch25/ (205 cases)")
print("All 205 Pooch25 cases are csPCa-positive by design.")
df_summary

Complete 1500-case set: resampled/ (1295 cases) + Pooch25/ (205 cases)
All 205 Pooch25 cases are csPCa-positive by design.


,folder,n_cases,in_plane_res,label_type,same_cases_as
0,original,1295,~2 mm (native),"ISUP granular (0,2-5)",resampled
1,resampled,1295,~0.3-0.5 mm (T2W),"ISUP granular (0,2-5)",original
2,Pooch25,205,~0.3-0.5 mm (T2W),"binary (0,1), all positive",none
